#  LexiLingo: Unified LoRA Adapter Fine-tuning (Multi-Task Learning)

**Version:** 3.0

**Mục đích:** Fine-tune Qwen3-1.7B với **1 unified LoRA adapter** để xử lý đồng thời 4 tasks:
1.  **Fluency Scoring** (0.0-1.0)
2.  **Vocabulary Level Classification** (A1, A2, B1, B2, C1, C2)
3.  **Grammar Error Correction** (GEC)
4.  **Dialogue Generation** (conversational responses)


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
# Install required packages (Colab-compatible)
# NOTE: protobuf must be >=5.26.1 but <6.0.0 for Colab compatibility
# NOTE: Pin datasets/trl to versions compatible with unsloth-zoo.
%pip install -q -U \
  "protobuf>=5.26.1,<6.0.0" \
  "pandas==2.2.2" \
  "transformers>=4.41.0" \
  "accelerate>=0.29.0" \
  "datasets>=3.4.1,<4.4.0,!=4.0.*,!=4.1.0" \
  "peft>=0.10.0" \
  "trl>=0.18.2,<=0.24.0,!=0.19.0" \
  "bitsandbytes>=0.43.1" \
  "sentencepiece" \
  "scipy" \
  "wandb" \
  "pymongo" \
  "matplotlib" \
  "seaborn" \
  "scikit-learn" \
  "jiwer" \
  "bert-score"

# Install Unsloth (optimized LoRA training - 2x faster, 70% less VRAM)
# Must install AFTER other packages to avoid dependency conflicts
%pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Optional but useful: verify no broken dependencies after install
!pip check | cat

In [ ]:
# QUAN TRỌNG: Mount Google Drive để lưu checkpoint và model
# Checkpoint sẽ được lưu vào Drive để không mất khi Colab disconnect
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Verify Drive is mounted
    drive_path = Path('/content/drive/MyDrive')
    if drive_path.exists():
        print("\n" + "="*70)
        print("GOOGLE DRIVE MOUNTED SUCCESSFULLY")
        print("="*70)
        print(f"Drive path: {drive_path}")
        print("\nCheckpoint will be saved to:")
        print(f"  /content/drive/MyDrive/LexiLingo/models/unified/")
        print("\nThis ensures data persists even if Colab disconnects!")
        print("="*70 + "\n")
    else:
        print("\nWARNING: Drive mount failed! Checkpoints will be lost on disconnect.")
        
except Exception as e:
    print(f"\nRunning locally (not Colab): {e}")
    print("Checkpoints will be saved to: ./model/outputs/unified/\n")

In [ ]:
# Verify installed versions and conflict-prone packages
import sys
import subprocess
print("Installed versions:")
subprocess.run([sys.executable, "-m", "pip", "show", "protobuf"], check=False)
subprocess.run([sys.executable, "-m", "pip", "show", "datasets"], check=False)
subprocess.run([sys.executable, "-m", "pip", "show", "trl"], check=False)
subprocess.run([sys.executable, "-m", "pip", "show", "unsloth-zoo"], check=False)
print("\nDependency check:")
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)
print("\nIf you see import errors, restart runtime (Runtime -> Restart runtime), then run from Cell 2.")

In [2]:
import torch
import json
import os
from pathlib import Path
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
    set_seed,
 )
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
 )
from trl import SFTTrainer
import numpy as np

# Reproducibility
set_seed(42)

# Device & precision (Colab GPU-first)
if torch.cuda.is_available():
    device = torch.device('cuda')
    major, minor = torch.cuda.get_device_capability(0)
    use_bf16 = major >= 8  # Ampere+
    use_fp16 = not use_bf16
    print(f" CUDA available: {torch.cuda.get_device_name(0)} (capability {major}.{minor})")
    print(f"Precision: {'bf16' if use_bf16 else 'fp16'}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    use_bf16 = False
    use_fp16 = False
    print(" MPS available (Apple Silicon)")
else:
    device = torch.device('cpu')
    use_bf16 = False
    use_fp16 = False
    print(" Running on CPU (slow). Consider Colab GPU.")

print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Set PyTorch memory allocator to avoid fragmentation (helps with OOM)
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print(" Set PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True")

In [ ]:
import json
from pathlib import Path
from datetime import datetime

class CheckpointManager:
    """Quản lý checkpoint và training state cho việc resume training"""
    
    def __init__(self, output_dir="./model/outputs/unified"):
        self.output_dir = Path(output_dir)
        self.state_file = self.output_dir / "training_state.json"
        self.output_dir.mkdir(parents=True, exist_ok=True)
    
    def find_latest_checkpoint(self):
        """Tìm checkpoint mới nhất"""
        checkpoints = sorted(
            [d for d in self.output_dir.glob("checkpoint-*") if d.is_dir()],
            key=lambda x: int(x.name.split("-")[-1])
        )
        return str(checkpoints[-1]) if checkpoints else None
    
    def list_all_checkpoints(self):
        """Liệt kê tất cả checkpoints"""
        checkpoints = sorted(
            [d for d in self.output_dir.glob("checkpoint-*") if d.is_dir()],
            key=lambda x: int(x.name.split("-")[-1])
        )
        return [{"path": str(cp), "step": int(cp.name.split("-")[-1])} for cp in checkpoints]
    
    def save_training_state(self, **kwargs):
        """Lưu thông tin training state"""
        state = {
            "last_update": datetime.now().isoformat(),
            **kwargs
        }
        with open(self.state_file, 'w') as f:
            json.dump(state, f, indent=2)
        print(f" Đã lưu training state: {self.state_file}")
    
    def load_training_state(self):
        """Load training state"""
        if self.state_file.exists():
            with open(self.state_file, 'r') as f:
                return json.load(f)
        return None
    
    def get_resume_info(self):
        """Lấy thông tin để resume training"""
        latest_checkpoint = self.find_latest_checkpoint()
        state = self.load_training_state()
        
        info = {
            "latest_checkpoint": latest_checkpoint,
            "has_checkpoint": latest_checkpoint is not None,
            "training_state": state,
            "all_checkpoints": self.list_all_checkpoints()
        }
        return info
    
    def print_status(self):
        """In ra trạng thái checkpoint"""
        info = self.get_resume_info()
        
        print("\n" + "="*70)
        print(" CHECKPOINT STATUS")
        print("="*70)
        
        if info["has_checkpoint"]:
            print(f" Tìm thấy {len(info['all_checkpoints'])} checkpoint(s)")
            print(f"\n Checkpoint mới nhất: {info['latest_checkpoint']}")
            
            if info["training_state"]:
                print(f"\n Training State:")
                for key, value in info["training_state"].items():
                    print(f"    {key}: {value}")
            
            print(f"\n Để resume training:")
            print(f"   resume_from_checkpoint='{info['latest_checkpoint']}'")
            print(f"   hoặc")
            print(f"   resume_from_checkpoint='auto'")
        else:
            print("  Chưa có checkpoint nào")
            print("   Training sẽ bắt đầu từ đầu")
        
        print("="*70 + "\n")
        
        return info

#  LƯU Ý: CheckpointManager sẽ được khởi tạo SAU KHI cấu hình OUTPUT_DIR
# (Xem cell Configuration bên dưới)
# Đảm bảo nó sử dụng đúng đường dẫn Drive hoặc local

# Kiểm tra class đã được định nghĩa
print(" CheckpointManager class ready")
print("   Sẽ được khởi tạo với OUTPUT_DIR từ configuration")

In [ ]:
import signal
import sys
import atexit
from datetime import datetime

class GracefulShutdownHandler:
    """
    Handler để tự động lưu checkpoint khi training bị ngắt đột ngột.
    
    Bắt các signal:
    - SIGINT: Ctrl+C (keyboard interrupt)
    - SIGTERM: System shutdown
    - atexit: Python process exit
    """
    
    def __init__(self):
        self.trainer = None
        self.model = None
        self.checkpoint_mgr = None
        self.emergency_save_path = None
        
        # Register signal handlers
        signal.signal(signal.SIGINT, self._signal_handler)
        signal.signal(signal.SIGTERM, self._signal_handler)
        atexit.register(self._emergency_save)
        
        print("  Graceful Shutdown Handler activated")
        print("    SIGINT (Ctrl+C): ")
        print("    SIGTERM (shutdown): ")
        print("    atexit (emergency): \n")
    
    def register_trainer(self, trainer, model, checkpoint_mgr):
        """Đăng ký trainer để có thể save khi cần"""
        self.trainer = trainer
        self.model = model
        self.checkpoint_mgr = checkpoint_mgr
        self.emergency_save_path = Path(trainer.args.output_dir) / "emergency_checkpoint"
        print(f" Trainer registered for auto-save")
        print(f"   Emergency path: {self.emergency_save_path}\n")
    
    def _signal_handler(self, signum, frame):
        """Xử lý khi nhận được signal ngắt"""
        signal_name = "SIGINT" if signum == signal.SIGINT else "SIGTERM"
        print(f"\n\n{'='*70}")
        print(f"  RECEIVED {signal_name} - Training interrupted!")
        print(f"{'='*70}\n")
        
        if self.trainer is not None and self.model is not None:
            try:
                print(" Emergency save in progress...")
                
                # Save checkpoint
                self.emergency_save_path.mkdir(parents=True, exist_ok=True)
                self.model.save_pretrained(str(self.emergency_save_path))
                
                # Save training state
                if self.checkpoint_mgr:
                    self.checkpoint_mgr.save_training_state(
                        status="interrupted",
                        signal=signal_name,
                        timestamp=datetime.now().isoformat(),
                        note=f"Training interrupted by {signal_name}",
                    )
                
                print(f" Emergency checkpoint saved to: {self.emergency_save_path}")
                print(f"   You can resume from this checkpoint later.\n")
                
            except Exception as e:
                print(f" Emergency save failed: {e}")
        else:
            print("  No trainer registered, cannot save checkpoint")
        
        print(f"{'='*70}\n")
        sys.exit(0)
    
    def _emergency_save(self):
        """Emergency save khi Python process exit"""
        # Only save if trainer exists and hasn't been saved yet
        if self.trainer is not None and self.model is not None:
            if not self.emergency_save_path or not self.emergency_save_path.exists():
                print("\n Emergency exit detected - attempting final save...")
                try:
                    self.emergency_save_path.mkdir(parents=True, exist_ok=True)
                    self.model.save_pretrained(str(self.emergency_save_path))
                    print(f" Final checkpoint saved to: {self.emergency_save_path}")
                except:
                    pass  # Silent fail in atexit

# Khởi tạo handler (run ngay khi load notebook)
shutdown_handler = GracefulShutdownHandler()

In [ ]:
#  Kiểm tra checkpoint trước khi config
# Cell này sẽ hiển thị có checkpoint nào để resume không
print("\n Checking for existing checkpoints before configuration...\n")

# Tạo output directory nếu chưa có
import os
from pathlib import Path

DRIVE_OUT = "/content/drive/MyDrive/LexiLingo/models"
LOCAL_OUT = "./model/outputs"  # Lưu vào folder model/outputs trong workspace
BASE_OUT = DRIVE_OUT if Path(DRIVE_OUT).exists() else LOCAL_OUT
OUTPUT_DIR_TEMP = str(Path(BASE_OUT) / "unified_lora_adapter")
Path(OUTPUT_DIR_TEMP).mkdir(parents=True, exist_ok=True)

print(f" Output directory: {OUTPUT_DIR_TEMP}")
print(f"   (Trong workspace: {Path(OUTPUT_DIR_TEMP).resolve()})\n")

# Khởi tạo checkpoint manager tạm để check
checkpoint_mgr_temp = CheckpointManager(OUTPUT_DIR_TEMP)
resume_info_temp = checkpoint_mgr_temp.get_resume_info()

if resume_info_temp["has_checkpoint"]:
    print(f"Found existing checkpoint: {resume_info_temp['latest_checkpoint']}")
    print(f"Training sẽ tự động resume từ checkpoint này")
else:
    print("No existing checkpoint found")

    print("Training sẽ bắt đầu từ đầu")
print("\n" + "="*70)

In [4]:
MODEL_NAME = 'Qwen/Qwen3-1.7B'
MAX_SEQ_LENGTH = 512

# Stability-first profile: prioritize smooth convergence over aggressive speed.
# Rationale from previous run: LR too high + fp16 can trigger NaN / loss spikes late in training.
UNIFIED_LORA_CONFIG = {
    'task_type': TaskType.CAUSAL_LM,
    'r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.08,
    'bias': 'none',
    'target_modules': [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    'inference_mode': False,
}

DRIVE_MOUNT = Path('/content/drive')
DRIVE_BASE = Path('/content/drive/MyDrive/LexiLingo')
LOCAL_BASE = Path('./model/outputs')

if DRIVE_MOUNT.exists():
    BASE_OUT = DRIVE_BASE
    print(' Google Drive detected -> using Drive for checkpoints/models')
else:
    BASE_OUT = LOCAL_BASE
    print(' Drive not detected -> using local output folder')

BASE_OUT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = str(BASE_OUT / 'unified_lora_adapter')
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

try:
    _t = Path(OUTPUT_DIR) / '.write_test'
    _t.write_text('ok')
    _t.unlink()
except Exception as e:
    raise RuntimeError(f'Không thể ghi vào output directory: {e}')

STABILITY_PROFILE = {
    'name': 'conservative-stable-v1',
    'goals': [
        'avoid NaN/Inf gradients in fp16',
        'improve monotonic eval behavior',
        'prefer best-checkpoint retention over long unstable runs',
    ],
}

if torch.cuda.is_available():
    TRAINING_CONFIG = {
        'output_dir': OUTPUT_DIR,
        'num_train_epochs': 4,
        'per_device_train_batch_size': 3,
        'per_device_eval_batch_size': 4,
        'gradient_accumulation_steps': 8,
        'learning_rate': 8e-5,
        'weight_decay': 0.03,
        'warmup_ratio': 0.10,
        'lr_scheduler_type': 'linear',
        'logging_steps': 10,
        'save_steps': 150,
        'eval_steps': 150,
        'save_total_limit': 3,
        'save_strategy': 'steps',
        'load_best_model_at_end': True,
        'fp16': bool(use_fp16),
        'bf16': bool(use_bf16),
        'gradient_checkpointing': True,
        'optim': 'paged_adamw_8bit',
        'adam_beta2': 0.95,
        'report_to': 'none',
        'dataloader_num_workers': 2,
        'max_grad_norm': 0.5,
        'eval_accumulation_steps': 8,
        'group_by_length': True,
    }
else:
    TRAINING_CONFIG = {
        'output_dir': OUTPUT_DIR,
        'num_train_epochs': 3,
        'per_device_train_batch_size': 2,
        'per_device_eval_batch_size': 2,
        'gradient_accumulation_steps': 12,
        'learning_rate': 6e-5,
        'weight_decay': 0.03,
        'warmup_ratio': 0.08,
        'lr_scheduler_type': 'linear',
        'logging_steps': 10,
        'save_steps': 120,
        'eval_steps': 120,
        'save_total_limit': 3,
        'save_strategy': 'steps',
        'load_best_model_at_end': True,
        'fp16': False,
        'bf16': False,
        'gradient_checkpointing': True,
        'optim': 'adamw_torch',
        'adam_beta2': 0.95,
        'report_to': 'none',
        'dataloader_num_workers': 2,
        'max_grad_norm': 0.5,
        'eval_accumulation_steps': 8,
        'group_by_length': True,
    }

checkpoint_mgr = CheckpointManager(TRAINING_CONFIG['output_dir'])

print('\n' + '='*70)
print('OPTIMIZED CONFIGURATION SUMMARY')
print('='*70)
print(f"Profile: {STABILITY_PROFILE['name']}")
print(f'Model: {MODEL_NAME}')
print('Base: Qwen3-1.7B')
print(f'MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}')
print(f"LoRA r/alpha/dropout: {UNIFIED_LORA_CONFIG['r']} / {UNIFIED_LORA_CONFIG['lora_alpha']} / {UNIFIED_LORA_CONFIG['lora_dropout']}")
print(f"Train batch x accum: {TRAINING_CONFIG['per_device_train_batch_size']} x {TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"LR / scheduler: {TRAINING_CONFIG['learning_rate']} / {TRAINING_CONFIG['lr_scheduler_type']}")
print(f"Max grad norm: {TRAINING_CONFIG['max_grad_norm']}")
print(f"Optimizer: {TRAINING_CONFIG['optim']} (beta2={TRAINING_CONFIG['adam_beta2']})")
print(f"Output directory: {TRAINING_CONFIG['output_dir']}")
print(f"Save/Eval steps: {TRAINING_CONFIG['save_steps']} / {TRAINING_CONFIG['eval_steps']}")
print(f"Precision: fp16={TRAINING_CONFIG['fp16']} bf16={TRAINING_CONFIG['bf16']}")
print('='*70 + '\n')

In [ ]:
# Training Speed Estimator - So sánh thời gian training cho Qwen3-1.7B

def estimate_training_time(config_name, seq_len, lora_r, batch_size, 
                          grad_accum, epochs, dataset_size=18400):
    """
    Ước tính thời gian training dựa trên hardware benchmarks.
    Mọi cấu hình trong notebook này đều dùng Qwen3-1.7B.
    """
    
    # Base time per sample trên T4 GPU (milliseconds) cho Qwen3-1.7B
    base_time_per_sample = 280  # ms
    
    # Adjustments
    time_ms = base_time_per_sample
    
    # Sequence length adjustment (quadratic for attention)
    seq_factor = (seq_len / 512) ** 1.5
    time_ms *= seq_factor
    
    # LoRA rank adjustment (more params = slower backward)
    lora_factor = 1 + (lora_r / 100)
    time_ms *= lora_factor
    
    # Batch size efficiency (larger batch = better GPU utilization)
    batch_efficiency = min(1.0, 0.5 + (batch_size / 8))
    time_ms *= (2 - batch_efficiency)
    
    # Calculate total
    effective_batch = batch_size * grad_accum
    steps_per_epoch = dataset_size / effective_batch
    total_steps = steps_per_epoch * epochs
    
    # Time per step = time_ms * batch_size (forward + backward + optimizer)
    time_per_step_sec = (time_ms * batch_size * grad_accum) / 1000
    
    total_time_sec = time_per_step_sec * total_steps
    hours = total_time_sec / 3600
    
    print(f"\n{'='*70}")
    print(f"  {config_name}")
    print(f"{'='*70}")
    print("Model: Qwen3-1.7B")
    print(f"  ├─ Sequence length: {seq_len}")
    print(f"  ├─ LoRA rank: {lora_r}")
    print(f"  ├─ Batch size: {batch_size} × {grad_accum} = {effective_batch}")
    print(f"  └─ Epochs: {epochs}")
    print(f"\nDataset: {dataset_size:,} samples")
    print(f"  ├─ Steps per epoch: {steps_per_epoch:.0f}")
    print(f"  └─ Total steps: {total_steps:.0f}")
    print(f"\nEstimated Time:")
    print(f"  ├─ Per step: {time_per_step_sec:.2f}s")
    print(f"  ├─ Per epoch: {(time_per_step_sec * steps_per_epoch / 60):.1f} min")
    print(f"  └─ Total: {hours:.2f} hours ({total_time_sec/60:.0f} minutes)")
    print(f"{'='*70}\n")
    
    return total_time_sec

# Compare two Qwen3-1.7B configs
print("\n TRAINING TIME COMPARISON (Qwen3-1.7B on T4 GPU)\n")

baseline_time = estimate_training_time(
    config_name=" BASELINE CONFIG (Qwen3-1.7B)",
    seq_len=768,
    lora_r=48,
    batch_size=2,
    grad_accum=12,
    epochs=7
)

optimized_time = estimate_training_time(
    config_name=" OPTIMIZED CONFIG (CURRENT GPU PROFILE)",
    seq_len=512,
    lora_r=16,
    batch_size=3,
    grad_accum=8,
    epochs=4
)

# Summary
speedup = baseline_time / optimized_time
time_saved = baseline_time - optimized_time

print("\n" + "="*70)
print("   PERFORMANCE IMPROVEMENT SUMMARY")
print("="*70)
print(f"Baseline config total time: {baseline_time/3600:.2f} hours")
print(f"Optimized config total time: {optimized_time/3600:.2f} hours")
print(f"\n Speedup: {speedup:.2f}x faster")
print(f" Time saved: {time_saved/3600:.2f} hours ({time_saved/60:.0f} minutes)")
print(f" Reduction: {((baseline_time - optimized_time) / baseline_time * 100):.1f}%")
print("="*70)

# Memory estimate (Qwen3-1.7B only)
print("\n" + "="*70)
print("   MEMORY USAGE ESTIMATE (Qwen3-1.7B, GPU)")
print("="*70)
print("Baseline config:")
print("  ├─ Model (4-bit): ~3.5 GB")
print("  ├─ Activations (seq=768, bs=2): ~3.0 GB")
print("  ├─ Optimizer states: ~2.0 GB")
print("  └─ Total: ~8.5 GB")
print("\nOptimized config:")
print("  ├─ Model (4-bit): ~3.5 GB")
print("  ├─ Activations (seq=512, bs=3): ~2.5 GB")
print("  ├─ Optimizer states: ~1.5 GB")
print("  └─ Total: ~7.5 GB")
print("\nEstimated memory reduction: ~12%")
print("="*70 + "\n")

In [5]:
# Load model with Unsloth optimization (up to 2x faster, 70% less memory)
from unsloth import FastLanguageModel
import torch

print('Loading model with Unsloth optimization...')

if torch.cuda.is_available():
    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        dtype=None,
    )

    base_model = FastLanguageModel.get_peft_model(
        base_model,
        r=UNIFIED_LORA_CONFIG['r'],
        target_modules=UNIFIED_LORA_CONFIG['target_modules'],
        lora_alpha=UNIFIED_LORA_CONFIG['lora_alpha'],
        lora_dropout=UNIFIED_LORA_CONFIG['lora_dropout'],
        bias=UNIFIED_LORA_CONFIG['bias'],
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )
    print('\nModel patched with Unsloth LoRA')
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    print('\nNo CUDA detected. Falling back to standard transformers loading...')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side='right')
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map='auto',
        low_cpu_mem_usage=True,
    )

assert 'qwen3' in MODEL_NAME.lower(), f'This notebook must use Qwen3, got: {MODEL_NAME}'

tokenizer.pad_token = tokenizer.eos_token
base_model.config.use_cache = False

actual_total_params_m = sum(p.numel() for p in base_model.parameters()) / 1e6

print(f'\nModel setup complete: {MODEL_NAME}')
print(f'Vocab size: {tokenizer.vocab_size}')
print(f'Max sequence length: {MAX_SEQ_LENGTH}')
print(f'LoRA Rank: {UNIFIED_LORA_CONFIG["r"]}')
print(f'Actual total params: {actual_total_params_m:.1f}M')

## 4. Prepare Training Data

### 4.1 Fluency Scoring Dataset

In [6]:
# ============================================================================
# Load train/val/test from correct Colab path (Drive-first, local fallback)
# ============================================================================
from pathlib import Path
import json
from datasets import load_dataset, Dataset

CANDIDATE_DATA_DIRS = [
    Path('/content/drive/MyDrive/LexiLingo/training_data'),
    Path('./LexiLingo/training_data'),
    Path('/content/drive/MyDrive/LexiLingo/trainning_data'),
    Path('./trainning_data'),
    Path('/content/drive/MyDrive/LexiLingo'),
    Path('./LexiLingo'),
    Path('/content/drive/MyDrive/LexiLingo/training_data/downloaded_datasets'),
    Path('/content/drive/MyDrive/LexiLingo/ai-service/model-development/datasets/datasets'),
    Path('./model-development/datasets/datasets'),
    Path('./datasets/datasets'),
    Path('./downloaded_datasets'),
]

data_dir = next((p for p in CANDIDATE_DATA_DIRS if p.exists()), None)
if data_dir is None:
    raise FileNotFoundError(
        'No dataset folder found. Checked:\n'
        + '\n'.join([f'  - {p}' for p in CANDIDATE_DATA_DIRS])
    )

print(f' Using dataset folder: {data_dir}')

report_path = data_dir / 'split_report.json'
if report_path.exists():
    rep = json.loads(report_path.read_text(encoding='utf-8'))
    print('\n split_report.json')
    print(json.dumps(rep.get('split', {}), indent=2, ensure_ascii=False))

train_jsonl = data_dir / 'train.jsonl'
val_jsonl = data_dir / 'val.jsonl'
test_jsonl = data_dir / 'test.jsonl'
unified_json = data_dir / 'unified_training_data.json'

if train_jsonl.exists() and val_jsonl.exists():
    print('\n Loading train/val JSONL...')
    train_raw = load_dataset('json', data_files=str(train_jsonl), split='train')
    val_raw = load_dataset('json', data_files=str(val_jsonl), split='train')

    if test_jsonl.exists():
        print(' Loading test JSONL...')
        test_raw = load_dataset('json', data_files=str(test_jsonl), split='train')
    else:
        test_raw = None
        print(' test.jsonl not found -> final test evaluation disabled in this run')

elif unified_json.exists():
    print('\n Fallback: unified_training_data.json -> split train/val/test (80/10/10)...')
    with open(unified_json, 'r', encoding='utf-8') as f:
        unified_training_data = json.load(f)

    raw = Dataset.from_list(unified_training_data)
    split_1 = raw.train_test_split(test_size=0.20, seed=42)
    train_raw = split_1['train']
    rem = split_1['test']
    split_2 = rem.train_test_split(test_size=0.50, seed=42)
    val_raw = split_2['train']
    test_raw = split_2['test']

else:
    raise FileNotFoundError('Missing train.jsonl/val.jsonl (or unified_training_data.json fallback)')

print(f' Train: {len(train_raw)}')
print(f' Val  : {len(val_raw)}')
print(f' Test : {len(test_raw) if test_raw is not None else 0}')

TASK_NAME_MAP = {
    'fluency': 'fluency_scoring',
    'vocabulary': 'vocabulary_classification',
    'grammar': 'grammar_correction',
    'dialogue': 'dialogue_response',
}

SYSTEM_PROMPT = (
    "You are LexiLingo's unified English tutor model. "
    "Follow the task instruction and respond ONLY with valid JSON (no extra text)."
)

def _safe_get(dct, *keys, default=None):
    cur = dct
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

def format_unified_prompt(example):
    """Format 1 record into Qwen chat template (messages-first, legacy fallback)."""
    task = example.get('task')
    messages = example.get('messages', [])
    if not isinstance(messages, list):
        messages = []
    normalized_messages = [m for m in messages if isinstance(m, dict) and 'role' in m and 'content' in m]

    if not normalized_messages:
        task_name = TASK_NAME_MAP.get(task, task or 'unknown')
        user_text = example.get('input', '')
        output_obj = example.get('output', {})
        metadata = example.get('metadata', {}) if isinstance(example.get('metadata'), dict) else {}

        if task_name in ('fluency_scoring', 'vocabulary_classification', 'grammar_correction'):
            prompt = f'Task: {task_name}\nText: {user_text}'
        else:
            history = metadata.get('history') or metadata.get('conversation_history') or ''
            strategy = _safe_get(output_obj, 'strategy') or metadata.get('strategy') or 'socratic_questioning'
            prompt = (
                f'Task: {task_name}\n'
                f'Text: {user_text}\n'
                f'Context: {history}\n'
                f'Strategy: {strategy}'
            )

        if isinstance(output_obj, (dict, list)):
            response = json.dumps(output_obj, ensure_ascii=False)
        else:
            response = json.dumps({'response': str(output_obj)}, ensure_ascii=False)

        normalized_messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
            {'role': 'assistant', 'content': response},
        ]

    text = tokenizer.apply_chat_template(
        normalized_messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return {'text': text, 'task': str(task or 'unknown')}

print('\n Formatting train/val/test with chat templates...')
train_dataset = train_raw.map(format_unified_prompt)
val_dataset = val_raw.map(format_unified_prompt)
test_dataset = test_raw.map(format_unified_prompt) if test_raw is not None else None

print(f' Train ready: {len(train_dataset)}')
print(f' Val ready:   {len(val_dataset)}')
print(f' Test ready:  {len(test_dataset) if test_dataset is not None else 0}')
print('\n Example formatted prompt:')
print('=' * 60)
print(train_dataset[0]['text'][:600] + '...')
print('=' * 60)

In [18]:
# MongoDB Logging Middleware (based on architecture.md)
import pymongo
from datetime import datetime
from typing import Dict, Any

# Safety: if cells executed out-of-order, ensure MONGODB_CONFIG exists
if "MONGODB_CONFIG" not in globals():
    MONGODB_CONFIG = {
        "enabled": False,
        "connection_string": "mongodb://localhost:27017/",
        "database": "lexilingo_training",
        "collections": {
            "training_logs": "training_logs",
            "model_metrics": "model_metrics",
            "training_queue": "training_queue",
        },
    }
    print(" MONGODB_CONFIG was not defined yet → using default (disabled)")

class MongoDBLogger:
    """
    Logging middleware for training metrics
    Stores training logs in MongoDB for analysis
    """
    def __init__(self, config: Dict[str, Any]):
        self.enabled = bool(config.get("enabled", False))
        if not self.enabled:
            print("  MongoDB logging disabled")
            return
        
        try:
            self.client = pymongo.MongoClient(
                config["connection_string"],
                serverSelectionTimeoutMS=5000,
            )
            self.db = self.client[config["database"]]
            
            # Collections
            self.training_logs = self.db[config["collections"]["training_logs"]]
            self.model_metrics = self.db[config["collections"]["model_metrics"]]
            self.training_queue = self.db[config["collections"]["training_queue"]]
            
            # Test connection
            self.client.server_info()
            print(" MongoDB connected successfully")
            
            # Create indexes
            self.training_logs.create_index([("timestamp", -1)])
            self.model_metrics.create_index([("epoch", 1)])
            
        except Exception as e:
            print(f"  MongoDB connection failed: {e}")
            self.enabled = False
    
    def log_training_step(self, step: int, loss: float, learning_rate: float, task: str = None):
        """Log training step metrics"""
        if not self.enabled:
            return
        
        try:
            self.training_logs.insert_one({
                "timestamp": datetime.now(),
                "step": step,
                "loss": float(loss),
                "learning_rate": float(learning_rate),
                "task": task,
                "model": MODEL_NAME,
            })
        except Exception as e:
            print(f"  Failed to log step: {e}")
    
    def log_epoch_metrics(self, epoch: int, metrics: Dict[str, float]):
        """Log epoch-level metrics"""
        if not self.enabled:
            return
        
        try:
            payload = {k: (float(v) if isinstance(v, (int, float)) else v) for k, v in metrics.items()}
            self.model_metrics.insert_one({
                "timestamp": datetime.now(),
                "epoch": int(epoch),
                "model": MODEL_NAME,
                **payload,
            })
        except Exception as e:
            print(f"  Failed to log epoch: {e}")
    
    def close(self):
        """Close MongoDB connection"""
        if getattr(self, "enabled", False):
            self.client.close()

# Initialize logger (won't crash if config missing)
mongo_logger = MongoDBLogger(MONGODB_CONFIG)

print("\n To enable MongoDB logging:")
print("  1) Set MONGODB_CONFIG['enabled'] = True")
print("  2) Provide a reachable MongoDB URI (Atlas or local)")
print("  3) Re-run this cell")

In [ ]:
def finetune_unified_adapter(train_dataset, eval_dataset, lora_config, resume_from_checkpoint=None):
    """Fine-tune unified LoRA adapter with task-aware eval metrics and stability safeguards."""
    from collections import defaultdict, deque

    print(f"\n{'='*60}")
    print('Training UNIFIED LoRA Adapter (Multi-Task Learning)')
    print(f"{'='*60}\n")

    def _select_stable_resume_checkpoint(auto_checkpoint):
        """Prefer best checkpoint if latest checkpoint shows instability signs."""
        if not auto_checkpoint:
            return None

        cp_path = Path(auto_checkpoint)
        state_path = cp_path / 'trainer_state.json'
        if not state_path.exists():
            return auto_checkpoint

        try:
            st = json.loads(state_path.read_text(encoding='utf-8'))
            logs = st.get('log_history', [])[-120:]
            has_non_finite = False
            for row in logs:
                if not isinstance(row, dict):
                    continue
                g = row.get('grad_norm')
                l = row.get('loss')
                if isinstance(g, float) and not np.isfinite(g):
                    has_non_finite = True
                    break
                if isinstance(l, float) and not np.isfinite(l):
                    has_non_finite = True
                    break

            best_cp = st.get('best_model_checkpoint')
            best_metric = st.get('best_metric')
            latest_eval = None
            for row in reversed(logs):
                if isinstance(row, dict) and 'eval_loss' in row:
                    latest_eval = row['eval_loss']
                    break

            eval_degraded = (
                isinstance(best_metric, (int, float))
                and isinstance(latest_eval, (int, float))
                and latest_eval > (best_metric * 1.12)
            )

            if (has_non_finite or eval_degraded) and best_cp:
                print('Stability check: latest checkpoint looks unstable -> resume from best checkpoint instead')
                print(f'  best_model_checkpoint={best_cp}')
                return best_cp

        except Exception as e:
            print(f'[Warning] Failed to inspect checkpoint stability: {e}')

        return auto_checkpoint

    if resume_from_checkpoint == 'auto':
        latest_checkpoint = checkpoint_mgr.find_latest_checkpoint()
        if latest_checkpoint:
            resume_from_checkpoint = _select_stable_resume_checkpoint(latest_checkpoint)
            print(f'Auto-selected checkpoint: {resume_from_checkpoint}')
        else:
            resume_from_checkpoint = None
            print('No checkpoint found, starting from scratch')

    if resume_from_checkpoint:
        print(f'Resuming from checkpoint: {resume_from_checkpoint}\n')

    peft_config = LoraConfig(**lora_config)
    from peft import PeftModel
    model = base_model

    if not isinstance(model, PeftModel):
        print('Model not pre-patched. Applying standard PEFT patching...')
        model = prepare_model_for_kbit_training(model)
        model = get_peft_model(model, peft_config)
    else:
        print('Model already has LoRA adapters (Unsloth pre-patched)')

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print('Model info:')
    print(f'  Trainable params: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)')
    print(f"  LoRA rank: {lora_config['r']}")
    print(f"  LoRA alpha: {lora_config['lora_alpha']}")

    task_values = [str(t) for t in train_dataset['task']] + [str(t) for t in eval_dataset['task']]
    task_labels = sorted(set(task_values))
    if 'unknown' not in task_labels:
        task_labels.append('unknown')
    task_to_id = {name: i for i, name in enumerate(task_labels)}
    id_to_task = {i: name for name, i in task_to_id.items()}

    eval_task_ids = [task_to_id.get(str(t), task_to_id['unknown']) for t in eval_dataset['task']]

    def tokenize_function(examples):
        # Dynamic padding is handled by data collator (faster + cleaner labels).
        return tokenizer(
            examples['text'],
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
        )

    print('\nTokenizing datasets...')
    train_tokenized = train_dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=train_dataset.column_names,
        desc='Tokenizing train',
    )
    eval_tokenized = eval_dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=eval_dataset.column_names,
        desc='Tokenizing eval',
    )

    ta_supported = set(TrainingArguments.__init__.__code__.co_varnames)
    training_kwargs = {
        'output_dir': TRAINING_CONFIG['output_dir'],
        'num_train_epochs': TRAINING_CONFIG['num_train_epochs'],
        'per_device_train_batch_size': TRAINING_CONFIG['per_device_train_batch_size'],
        'per_device_eval_batch_size': TRAINING_CONFIG['per_device_eval_batch_size'],
        'gradient_accumulation_steps': TRAINING_CONFIG['gradient_accumulation_steps'],
        'learning_rate': TRAINING_CONFIG['learning_rate'],
        'weight_decay': TRAINING_CONFIG['weight_decay'],
        'warmup_ratio': TRAINING_CONFIG['warmup_ratio'],
        'lr_scheduler_type': TRAINING_CONFIG['lr_scheduler_type'],
        'logging_steps': TRAINING_CONFIG['logging_steps'],
        'save_steps': TRAINING_CONFIG['save_steps'],
        'eval_steps': TRAINING_CONFIG['eval_steps'],
        'save_total_limit': TRAINING_CONFIG['save_total_limit'],
        'fp16': TRAINING_CONFIG['fp16'],
        'bf16': TRAINING_CONFIG['bf16'],
        'gradient_checkpointing': TRAINING_CONFIG['gradient_checkpointing'],
        'max_grad_norm': TRAINING_CONFIG['max_grad_norm'],
        'optim': TRAINING_CONFIG['optim'],
        'report_to': TRAINING_CONFIG['report_to'],
        'disable_tqdm': True,
        'logging_first_step': True,
        'save_strategy': 'steps',
        'load_best_model_at_end': True,
        # Multi-task focus: choose checkpoint by macro task token accuracy
        'metric_for_best_model': 'eval_token_accuracy_macro_tasks',
        'greater_is_better': True,
        'dataloader_num_workers': TRAINING_CONFIG['dataloader_num_workers'],
        'dataloader_pin_memory': torch.cuda.is_available(),
        'save_on_each_node': False,
    }
    if 'eval_strategy' in ta_supported:
        training_kwargs['eval_strategy'] = 'steps'
    else:
        training_kwargs['evaluation_strategy'] = 'steps'

    if 'save_safetensors' in ta_supported:
        training_kwargs['save_safetensors'] = True
    else:
        print('⚠ transformers quá cũ: bỏ qua save_safetensors')

    if 'adam_beta2' in ta_supported and 'adam_beta2' in TRAINING_CONFIG:
        training_kwargs['adam_beta2'] = TRAINING_CONFIG['adam_beta2']

    if 'group_by_length' in ta_supported:
        training_kwargs['group_by_length'] = bool(TRAINING_CONFIG.get('group_by_length', False))
    else:
        print('⚠ transformers quá cũ: bỏ qua group_by_length')

    if 'eval_accumulation_steps' in ta_supported:
        training_kwargs['eval_accumulation_steps'] = int(TRAINING_CONFIG.get('eval_accumulation_steps', 8))
    else:
        print('⚠ transformers quá cũ: bỏ qua eval_accumulation_steps')

    if 'logging_nan_inf_filter' in ta_supported:
        training_kwargs['logging_nan_inf_filter'] = False

    training_args = TrainingArguments(**training_kwargs)

    print('\nDataset:')
    print(f'  Train: {len(train_tokenized)} samples')
    print(f'  Eval : {len(eval_tokenized)} samples')
    print(f'  Tasks: {task_labels}')

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
        pad_to_multiple_of=8,
    )

    from transformers import TrainerCallback, EarlyStoppingCallback

    class CheckpointCallback(TrainerCallback):
        def on_save(self, args, state, control, **kwargs):
            checkpoint_mgr.save_training_state(
                epoch=state.epoch,
                global_step=state.global_step,
                best_metric=state.best_metric,
                best_model_checkpoint=state.best_model_checkpoint,
                total_epochs=args.num_train_epochs,
                learning_rate=args.learning_rate,
            )
            print(f'Checkpoint saved at step {state.global_step} (epoch {state.epoch:.2f})')

    class EvalPrintCallback(TrainerCallback):
        def on_evaluate(self, args, state, control, metrics=None, **kwargs):
            if not metrics:
                return
            print(f"\n[Eval] step={state.global_step} epoch={state.epoch:.2f}")
            if 'eval_loss' in metrics:
                print(f"  eval_loss={metrics['eval_loss']:.4f}")
            if 'eval_token_accuracy' in metrics:
                print(f"  token_acc={metrics['eval_token_accuracy']:.4f}")
            if 'eval_token_accuracy_macro_tasks' in metrics:
                print(f"  macro_task_acc={metrics['eval_token_accuracy_macro_tasks']:.4f}")
            task_keys = sorted(k for k in metrics if k.startswith('eval_task_') and k.endswith('_token_acc'))
            for k in task_keys:
                print(f"  {k.replace('eval_', '')}={metrics[k]:.4f}")

    class StabilityGuardCallback(TrainerCallback):
        """Guardrail for fp16/QLoRA instability: reduce LR on spikes, stop on repeated non-finite."""

        def __init__(self, non_finite_patience=2, spike_factor=4.0, spike_patience=3, lr_decay=0.5):
            self.non_finite_patience = non_finite_patience
            self.spike_factor = spike_factor
            self.spike_patience = spike_patience
            self.lr_decay = lr_decay
            self.non_finite_count = 0
            self.spike_count = 0
            self.loss_window = deque(maxlen=30)

        def _reduce_lr(self, optimizer):
            if optimizer is None:
                return
            for pg in optimizer.param_groups:
                pg['lr'] = max(pg['lr'] * self.lr_decay, 1e-6)

        def on_log(self, args, state, control, logs=None, **kwargs):
            if not logs:
                return

            optimizer = kwargs.get('optimizer')
            loss = logs.get('loss', None)
            grad_norm = logs.get('grad_norm', None)

            if isinstance(loss, (int, float)):
                loss = float(loss)
            else:
                loss = None

            if isinstance(grad_norm, (int, float)):
                grad_norm = float(grad_norm)
            else:
                grad_norm = None

            has_non_finite = False
            if loss is not None and not np.isfinite(loss):
                has_non_finite = True
            if grad_norm is not None and not np.isfinite(grad_norm):
                has_non_finite = True

            if has_non_finite:
                self.non_finite_count += 1
                print(f"[StabilityGuard] non-finite detected at step {state.global_step} ({self.non_finite_count}/{self.non_finite_patience})")
                self._reduce_lr(optimizer)
                if self.non_finite_count >= self.non_finite_patience:
                    print('[StabilityGuard] stopping training to protect best checkpoint.')
                    control.should_training_stop = True
                return
            else:
                self.non_finite_count = 0

            if loss is not None and np.isfinite(loss):
                baseline = float(np.median(self.loss_window)) if len(self.loss_window) >= 8 else None
                self.loss_window.append(loss)

                if baseline is not None and state.global_step > 80 and loss > baseline * self.spike_factor:
                    self.spike_count += 1
                    self._reduce_lr(optimizer)
                    print(
                        f"[StabilityGuard] loss spike at step {state.global_step}: "
                        f"loss={loss:.4f} baseline={baseline:.4f} -> LR reduced"
                    )
                    if self.spike_count >= self.spike_patience:
                        print('[StabilityGuard] repeated spikes, stopping training early.')
                        control.should_training_stop = True
                else:
                    if self.spike_count > 0:
                        self.spike_count -= 1

    def compute_metrics_fn(eval_pred):
        try:
            preds, labels = eval_pred

            if isinstance(preds, (tuple, list)):
                preds = preds[0]

            preds = np.asarray(preds)
            labels = np.asarray(labels)

            # Fallback safety: some transformers versions may still pass full logits.
            if preds.ndim == 3:
                preds = preds.argmax(-1)

            preds = np.nan_to_num(preds, nan=0.0, posinf=0.0, neginf=0.0).astype(np.int64, copy=False)
            labels = labels.astype(np.int64, copy=False)

            pad_id = tokenizer.pad_token_id
            if pad_id is None:
                pad_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0

            vocab_size = getattr(tokenizer, 'vocab_size', None)
            if not vocab_size or vocab_size <= 0:
                vocab_size = int(np.max(preds)) + 1 if preds.size else 1

            preds = np.where((preds < 0) | (preds >= vocab_size), pad_id, preds)

            n_rows = min(preds.shape[0], labels.shape[0])
            n_cols = min(preds.shape[1], labels.shape[1]) if preds.ndim == 2 and labels.ndim == 2 else 0
            if n_rows == 0 or n_cols < 2:
                return {'token_accuracy': 0.0}

            preds = preds[:n_rows, :n_cols]
            labels = labels[:n_rows, :n_cols]

            # Causal LM alignment: token at t predicts label at t+1
            preds_shift = preds[:, :-1]
            labels_shift = labels[:, 1:]

            valid_mask = labels_shift != -100
            if valid_mask.any():
                token_acc = float((preds_shift[valid_mask] == labels_shift[valid_mask]).mean())
            else:
                token_acc = 0.0

            metrics = {'token_accuracy': round(token_acc, 4)}

            sample_token_acc = []
            for i in range(n_rows):
                row_mask = labels_shift[i] != -100
                if np.any(row_mask):
                    row_acc = float((preds_shift[i][row_mask] == labels_shift[i][row_mask]).mean())
                else:
                    row_acc = 0.0
                sample_token_acc.append(row_acc)

            task_bucket = defaultdict(list)
            unknown_id = task_to_id['unknown']
            for i, acc in enumerate(sample_token_acc):
                tid = eval_task_ids[i] if i < len(eval_task_ids) else unknown_id
                task_bucket[tid].append(acc)

            task_means = []
            task_weighted_sum = 0.0
            task_total = 0
            for tid, values in task_bucket.items():
                task_name = id_to_task.get(tid, 'unknown')
                task_acc = float(np.mean(values)) if values else 0.0
                metrics[f'task_{task_name}_token_acc'] = round(task_acc, 4)
                task_means.append(task_acc)
                task_weighted_sum += task_acc * len(values)
                task_total += len(values)

            if task_means:
                # Macro task accuracy is more robust for imbalanced task distributions.
                metrics['token_accuracy_macro_tasks'] = round(float(np.mean(task_means)), 4)
            if task_total > 0:
                metrics['token_accuracy_weighted_tasks'] = round(float(task_weighted_sum / task_total), 4)

            return metrics
        except Exception as e:
            print(f'[Warning] compute_metrics failed: {type(e).__name__}: {e}')
            return {'token_accuracy': 0.0}

    def preprocess_logits(logits, labels):
        if isinstance(logits, tuple):
            logits = logits[0]
        return logits.argmax(-1)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=eval_tokenized,
        data_collator=data_collator,
        compute_metrics=compute_metrics_fn,
        preprocess_logits_for_metrics=preprocess_logits,
        callbacks=[
            CheckpointCallback(),
            EvalPrintCallback(),
            StabilityGuardCallback(non_finite_patience=2, spike_factor=4.0, spike_patience=3, lr_decay=0.5),
            EarlyStoppingCallback(early_stopping_patience=4, early_stopping_threshold=1e-4),
        ],
    )

    print('\nStarting training...')
    if torch.cuda.is_available():
        print(f"  Device: GPU ({torch.cuda.get_device_name(0)})")
    else:
        print('  Device: CPU/MPS (slow)')

    checkpoint_mgr.save_training_state(
        status='training_started',
        num_train_epochs=training_args.num_train_epochs,
        train_samples=len(train_tokenized),
        eval_samples=len(eval_tokenized),
        tasks=task_labels,
    )

    shutdown_handler.register_trainer(trainer, model, checkpoint_mgr)

    try:
        trainer.train(resume_from_checkpoint=resume_from_checkpoint)
    except KeyboardInterrupt:
        print('\nTraining interrupted by user (Ctrl+C)')
        print('Checkpoint already saved by shutdown handler')
        raise

    def _latest_eval_from_log_history(state):
        for row in reversed(getattr(state, 'log_history', [])):
            if isinstance(row, dict) and any(k.startswith('eval_') for k in row.keys()):
                return row
        return {}

    print('\nFinal evaluation:')
    try:
        eval_results = trainer.evaluate()
    except RuntimeError as e:
        if 'on_train_begin must be called before on_evaluate' in str(e):
            print('Notebook callback evaluate bug detected; falling back to latest eval metrics from log_history.')
            eval_results = _latest_eval_from_log_history(trainer.state)
        else:
            raise
    important_keys = [k for k in eval_results if k in {'eval_loss', 'eval_token_accuracy'} or k.startswith('eval_task_') or k.startswith('eval_token_accuracy_')]
    for key in sorted(important_keys):
        value = eval_results[key]
        if isinstance(value, (int, float)):
            print(f'  {key}: {value:.4f}')
        else:
            print(f'  {key}: {value}')

    adapter_path = str(Path(TRAINING_CONFIG['output_dir']) / 'unified_lora_adapter')
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    print(f'\nUnified LoRA adapter saved to: {adapter_path}')

    checkpoint_mgr.save_training_state(
        status='training_completed',
        final_eval_loss=eval_results.get('eval_loss'),
        final_eval_token_accuracy=eval_results.get('eval_token_accuracy'),
        final_eval_token_accuracy_macro_tasks=eval_results.get('eval_token_accuracy_macro_tasks'),
        adapter_path=adapter_path,
    )

    return model, trainer

print('\n' + '='*70)
print('TRAINING UNIFIED ADAPTER')
print('='*70)
print('Resume mode: auto (will auto-detect latest checkpoint)')
print('='*70 + '\n')

unified_model, trainer = finetune_unified_adapter(
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    lora_config=UNIFIED_LORA_CONFIG,
    resume_from_checkpoint='auto',
)

In [ ]:
#  Utility: Quản lý checkpoints (xóa cũ, chọn checkpoint cụ thể)

def list_checkpoints():
    """Liệt kê tất cả checkpoints với thông tin chi tiết"""
    checkpoints = checkpoint_mgr.list_all_checkpoints()
    
    if not checkpoints:
        print("  Không có checkpoint nào")
        return []
    
    print(f"\n Có {len(checkpoints)} checkpoint(s):\n")
    for i, cp in enumerate(checkpoints, 1):
        size = sum(f.stat().st_size for f in Path(cp['path']).rglob('*') if f.is_file())
        size_mb = size / (1024 * 1024)
        print(f"{i}. Step {cp['step']:,} - {cp['path']}")
        print(f"   Size: {size_mb:.1f} MB\n")
    
    return checkpoints

def remove_checkpoint(checkpoint_path):
    """Xóa một checkpoint cụ thể"""
    import shutil
    path = Path(checkpoint_path)
    if path.exists() and path.is_dir():
        shutil.rmtree(path)
        print(f" Đã xóa: {checkpoint_path}")
    else:
        print(f"  Không tìm thấy: {checkpoint_path}")

def clean_old_checkpoints(keep_last_n=2):
    """Giữ lại n checkpoints mới nhất, xóa các checkpoints cũ"""
    checkpoints = checkpoint_mgr.list_all_checkpoints()
    
    if len(checkpoints) <= keep_last_n:
        print(f"ℹ  Chỉ có {len(checkpoints)} checkpoint(s), không cần cleanup")
        return
    
    to_remove = checkpoints[:-keep_last_n]
    print(f"  Sẽ xóa {len(to_remove)} checkpoint(s) cũ, giữ lại {keep_last_n} checkpoint mới nhất\n")
    
    for cp in to_remove:
        remove_checkpoint(cp['path'])
    
    print(f"\n Cleanup hoàn tất!")

# List checkpoints hiện có
list_checkpoints()

# Uncomment dòng dưới để cleanup (giữ lại 2 checkpoints mới nhất)
# clean_old_checkpoints(keep_last_n=2)

In [ ]:
# VERIFY: Checkpoint đã được lưu vào Drive chưa?
import os
from pathlib import Path

def verify_checkpoint_location():
    """Kiểm tra và hiển thị vị trí checkpoint đã lưu"""
    print("\n" + "="*70)
    print("CHECKPOINT VERIFICATION")
    print("="*70)
    
    output_dir = Path(TRAINING_CONFIG['output_dir'])
    drive_root = Path("/content/drive/MyDrive/LexiLingo")
    
    print(f"\n1. Output directory configured:")
    print(f"   {output_dir}")
    print(f"   Resolved: {output_dir.resolve()}")
    
    print(f"\n2. Directory exists: {'YES' if output_dir.exists() else 'NO'}")
    
    if output_dir.exists():
        # List contents
        contents = list(output_dir.iterdir())
        print(f"\n3. Contents ({len(contents)} items):")
        
        checkpoints = [f for f in contents if f.is_dir() and f.name.startswith('checkpoint-')]
        adapters = [f for f in contents if f.is_dir() and 'adapter' in f.name.lower()]
        other_files = [f for f in contents if f.is_file()]
        
        if checkpoints:
            print(f"\n   Checkpoints found: {len(checkpoints)}")
            for cp in sorted(checkpoints):
                size = sum(f.stat().st_size for f in cp.rglob('*') if f.is_file())
                print(f"   - {cp.name} ({size / (1024**2):.1f} MB)")
        
        if adapters:
            print(f"\n   Adapters found: {len(adapters)}")
            for ad in adapters:
                size = sum(f.stat().st_size for f in ad.rglob('*') if f.is_file())
                print(f"   - {ad.name} ({size / (1024**2):.1f} MB)")
        
        if other_files:
            print(f"\n   Other files: {len(other_files)}")
            for f in other_files:
                print(f"   - {f.name} ({f.stat().st_size / 1024:.1f} KB)")
        
        # Check if on Drive
        if str(output_dir).startswith(str(drive_root)):
            print(f"\n4. Storage location: GOOGLE DRIVE")
            print(f"   Data is PERSISTENT and safe from Colab disconnects!")
            
            # Provide Drive link
            relative_path = str(output_dir).replace('/content/drive/MyDrive/', '')
            print(f"\n5. Access from Drive:")
            print(f"   Go to: My Drive > {relative_path}")
        else:
            print(f"\n4. Storage location: LOCAL (/content/)")
            print(f"   WARNING: Will be DELETED when Colab session ends!")
            print(f"\n   To save to Drive:")
            print(f"   1. Mount Drive (run Drive mount cell)")
            print(f"   2. Re-run training cell")
    else:
        print(f"\n   ERROR: Output directory not found!")
        print(f"   Training may not have started or failed.")
    
    print("="*70 + "\n")

# Run verification
verify_checkpoint_location()

## 5. Visualization - Training Metrics

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
import pandas as pd
from pathlib import Path

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print(" Visualization libraries loaded")
print(" Available visualizations:")
print("  1. Training loss curves")
print("  2. Task distribution")
print("  3. Model architecture summary")
print("  4. Parameter comparison")
print("  5. Evaluation metrics dashboard")

In [ ]:
# Visualize 1: Training Loss Curves
import numpy as np

def _moving_average(values, window=5):
    if len(values) < 2 or window <= 1:
        return np.array(values, dtype=float)
    kernel = np.ones(window, dtype=float) / window
    return np.convolve(np.array(values, dtype=float), kernel, mode='same')

def plot_training_loss(trainer, save_path=None, export_clean_plot=True):
    """Plot training/validation loss and export a clean model-loss chart like publication style."""
    if not hasattr(trainer, 'state') or not trainer.state.log_history:
        print(" No training history available. Train the model first!")
        return

    train_loss, eval_loss = [], []
    steps, eval_steps = [], []

    for log in trainer.state.log_history:
        if 'loss' in log and 'step' in log:
            v = float(log['loss'])
            if np.isfinite(v):
                train_loss.append(v)
                steps.append(int(log['step']))
        if 'eval_loss' in log and 'step' in log:
            v = float(log['eval_loss'])
            if np.isfinite(v):
                eval_loss.append(v)
                eval_steps.append(int(log['step']))

    if not train_loss:
        print(" No valid training loss values found in log_history.")
        return

    # Plot A: raw training progress (existing style)
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    ax.plot(steps, train_loss, 'b-', linewidth=2, label='Training Loss', alpha=0.8)
    if eval_loss:
        ax.plot(eval_steps, eval_loss, 'r-', linewidth=2, label='Validation Loss', alpha=0.8)

    ax.set_xlabel('Training Steps', fontsize=12, fontweight='bold')
    ax.set_ylabel('Loss', fontsize=12, fontweight='bold')
    ax.set_title('Training Progress - Unified LoRA Adapter', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    min_loss = min(train_loss)
    min_step = steps[train_loss.index(min_loss)]
    ax.axhline(y=min_loss, color='g', linestyle='--', alpha=0.5)
    ax.annotate(
        f'Min: {min_loss:.4f}\nStep: {min_step}',
        xy=(min_step, min_loss),
        xytext=(10, 10),
        textcoords='offset points',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7),
        fontsize=9,
    )

    plt.tight_layout()
    if save_path:
        import os
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f' Saved raw training curve to: {save_path}')
    plt.show()

    # Plot B: clean model-loss chart (like image 2)
    if export_clean_plot:
        train_arr = np.array(train_loss, dtype=float)
        eval_arr = np.array(eval_loss, dtype=float) if eval_loss else np.array([], dtype=float)

        # Clip extreme spikes for readability while keeping trend
        all_vals = train_arr.copy()
        if eval_arr.size > 0:
            all_vals = np.concatenate([all_vals, eval_arr])
        q95 = float(np.quantile(all_vals, 0.95)) if all_vals.size > 0 else 1.0
        upper = max(q95 * 1.25, 0.2)

        train_clean = np.clip(train_arr, 0.0, upper)
        eval_clean = np.clip(eval_arr, 0.0, upper) if eval_arr.size > 0 else eval_arr

        train_smooth = _moving_average(train_clean, window=5)
        eval_smooth = _moving_average(eval_clean, window=3) if eval_clean.size > 1 else eval_clean

        # Use normalized x-axis for cleaner "epoch-like" view
        x_train = np.arange(len(train_smooth))
        x_eval = np.linspace(0, len(train_smooth) - 1, num=len(eval_smooth)) if len(eval_smooth) > 0 else np.array([])

        fig2, ax2 = plt.subplots(figsize=(8, 6))
        ax2.plot(x_train, train_smooth, label='training', linewidth=2, color='#1f77b4')
        if len(eval_smooth) > 0:
            ax2.plot(x_eval, eval_smooth, label='validation', linewidth=2, color='#ff7f0e')

        ax2.set_title('Model Loss', fontsize=22, pad=12)
        ax2.set_xlabel('Epoch', fontsize=18)
        ax2.set_ylabel('Loss', fontsize=18)
        ax2.tick_params(axis='both', labelsize=12)
        ax2.legend(fontsize=12, loc='upper left')
        ax2.grid(False)

        y_max = max(float(np.max(train_smooth)), float(np.max(eval_smooth)) if len(eval_smooth) > 0 else 0.0)
        y_top = max(0.3, y_max * 1.08)
        ax2.set_ylim(0, y_top)

        plt.tight_layout()

        clean_path = None
        if save_path:
            p = Path(save_path)
            clean_path = str(p.with_name('model_loss_clean.png'))
            plt.savefig(clean_path, dpi=300, bbox_inches='tight')
            print(f' Saved clean model-loss chart to: {clean_path}')

        plt.show()

    print("\n Training Statistics:")
    print(f"  Total steps: {steps[-1] if steps else 0}")
    print(f"  Initial loss: {train_loss[0]:.4f}")
    print(f"  Final loss: {train_loss[-1]:.4f}")
    print(f"  Min loss: {min_loss:.4f}")
    if len(train_loss) > 1 and train_loss[0] != 0:
        print(f"  Loss reduction: {((train_loss[0] - train_loss[-1]) / train_loss[0] * 100):.1f}%")

# Auto-render loss chart if training already ran
if 'trainer' in globals() and trainer is not None:
    save_path = None
    if 'TRAINING_CONFIG' in globals() and isinstance(TRAINING_CONFIG, dict):
        save_path = str(Path(TRAINING_CONFIG['output_dir']) / 'training_loss_curve.png')
    plot_training_loss(trainer, save_path=save_path, export_clean_plot=True)
else:
    print('Trainer not found yet. Run training cell first, then re-run this cell to view loss chart.')

In [ ]:
# Visualization Dashboard: Core Training Results
from pathlib import Path
import json
import pandas as pd

def _latest_eval_metrics_from_trainer(trainer_obj):
    """Extract latest eval_* metrics from trainer state log history."""
    if trainer_obj is None or not hasattr(trainer_obj, 'state'):
        return {}
    for row in reversed(getattr(trainer_obj.state, 'log_history', [])):
        if isinstance(row, dict) and any(str(k).startswith('eval_') for k in row.keys()):
            return row
    return {}

def show_visualization_dashboard(trainer_obj=None, training_config=None):
    print("=" * 80)
    print("VISUALIZATION DASHBOARD - CORE RESULTS")
    print("=" * 80)

    if trainer_obj is None:
        print("Trainer chưa có trong memory. Hãy chạy cell training trước.")
        print("Sau đó chạy lại cell này để xem các chỉ số tổng hợp.")
        return

    latest_eval = _latest_eval_metrics_from_trainer(trainer_obj)
    important_eval_keys = [
        'eval_loss',
        'eval_token_accuracy',
        'eval_token_accuracy_macro_tasks',
        'eval_token_accuracy_weighted_tasks',
    ]

    print("\n[1] Latest Evaluation Metrics")
    if latest_eval:
        shown = 0
        for k in important_eval_keys:
            if k in latest_eval:
                v = latest_eval[k]
                if isinstance(v, (int, float)):
                    print(f"  - {k}: {v:.4f}")
                else:
                    print(f"  - {k}: {v}")
                shown += 1
        if shown == 0:
            print("  Có eval log nhưng chưa chứa các key chính cần hiển thị.")
    else:
        print("  Chưa tìm thấy eval_* trong trainer.state.log_history.")

    print("\n[2] Latest Training Loss Snapshot")
    train_loss_rows = []
    if hasattr(trainer_obj, 'state'):
        for row in getattr(trainer_obj.state, 'log_history', []):
            if isinstance(row, dict) and 'loss' in row:
                train_loss_rows.append(row)

    if train_loss_rows:
        last_loss = float(train_loss_rows[-1]['loss'])
        first_loss = float(train_loss_rows[0]['loss'])
        print(f"  - First loss: {first_loss:.4f}")
        print(f"  - Last loss : {last_loss:.4f}")
        if first_loss != 0:
            reduction = (first_loss - last_loss) / first_loss * 100
            print(f"  - Loss reduction: {reduction:.2f}%")
    else:
        print("  Chưa có log training loss.")

    print("\n[3] Test Report Status")
    if training_config is not None and isinstance(training_config, dict):
        report_dir = Path(training_config['output_dir']) / 'test_report'
        metrics_path = report_dir / 'test_metrics.json'
        if metrics_path.exists():
            print(f"  - Found: {metrics_path}")
            try:
                metrics_data = json.loads(metrics_path.read_text(encoding='utf-8'))
                preview = {}
                if 'vocabulary' in metrics_data and isinstance(metrics_data['vocabulary'], dict):
                    preview['vocabulary_F1_macro'] = metrics_data['vocabulary'].get('F1_macro')
                    preview['vocabulary_Balanced_Accuracy'] = metrics_data['vocabulary'].get('Balanced_Accuracy')
                if 'grammar' in metrics_data and isinstance(metrics_data['grammar'], dict):
                    preview['grammar_WER'] = metrics_data['grammar'].get('WER')
                if 'dialogue' in metrics_data and isinstance(metrics_data['dialogue'], dict):
                    preview['dialogue_BERTScore_F1'] = metrics_data['dialogue'].get('BERTScore_F1')

                if preview:
                    print("  - Key test metrics preview:")
                    for k, v in preview.items():
                        if isinstance(v, (int, float)):
                            print(f"      {k}: {v:.4f}")
                        else:
                            print(f"      {k}: {v}")
                else:
                    print("  - test_metrics.json có sẵn nhưng chưa có các key preview.")
            except Exception as e:
                print(f"  - Không đọc được test_metrics.json: {e}")
        else:
            print("  - Chưa có test_report/test_metrics.json.")
            print("    Hãy chạy cell Final test evaluation để tạo báo cáo.")
    else:
        print("  - TRAINING_CONFIG chưa sẵn sàng.")

    print("=" * 80)

# Auto-run dashboard
if 'trainer' in globals() and trainer is not None:
    cfg = TRAINING_CONFIG if 'TRAINING_CONFIG' in globals() else None
    show_visualization_dashboard(trainer_obj=trainer, training_config=cfg)
else:
    print("Trainer not found yet. Run training cell first, then run this dashboard cell.")

In [ ]:
# Visualize 2: Task Distribution
def plot_task_distribution(dataset, save_path=None):
    """Visualize distribution of tasks in training dataset"""
    task_counts = {}
    for item in dataset:
        task = item.get('task', 'unknown')
        task_counts[task] = task_counts.get(task, 0) + 1

    if not task_counts:
        print("  Dataset rỗng, không thể vẽ task distribution.")
        return

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    colors = ['#4285F4', '#34A853', '#FBBC04', '#EA4335']
    ax1.pie(
        task_counts.values(),
        labels=[f'{k.capitalize()}\n({v} samples)' for k, v in task_counts.items()],
        colors=colors,
        autopct='%1.1f%%',
        startangle=90,
        textprops={'fontsize': 11, 'fontweight': 'bold'}
    )
    ax1.set_title('Task Distribution (Pie Chart)', fontsize=14, fontweight='bold')

    tasks = list(task_counts.keys())
    counts = list(task_counts.values())
    bars = ax2.bar(tasks, counts, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

    for bar, count in zip(bars, counts):
        height = bar.get_height()
        ax2.text(
            bar.get_x() + bar.get_width() / 2.,
            height,
            f'{count}\n({count / sum(counts) * 100:.1f}%)',
            ha='center',
            va='bottom',
            fontweight='bold',
            fontsize=10
        )

    ax2.set_xlabel('Task Type', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Number of Samples', fontsize=12, fontweight='bold')
    ax2.set_title('Task Distribution (Bar Chart)', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.set_xticklabels([t.capitalize() for t in tasks])

    plt.tight_layout()
    if save_path:
        import os
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f' Saved plot to: {save_path}')
    plt.show()

    total = sum(task_counts.values())
    print("\n Dataset Composition:")
    print(f"  Total samples: {total}")
    for task, count in sorted(task_counts.items()):
        print(f"  {task.capitalize()}: {count} ({count / total * 100:.1f}%)")

    if len(set(task_counts.values())) == 1:
        print("\n Dataset is perfectly balanced!")
    else:
        max_count = max(task_counts.values())
        min_count = min(task_counts.values())
        ratio = max_count / min_count
        print(f"\n  Imbalance ratio: {ratio:.2f}x (max/min)")
        if ratio > 2:
            print("   Consider balancing tasks for better multi-task learning")

# Auto-render when train_raw is available
if 'train_raw' in globals() and train_raw is not None and len(train_raw) > 0:
    auto_save = None
    if 'TRAINING_CONFIG' in globals() and isinstance(TRAINING_CONFIG, dict):
        auto_save = str(Path(TRAINING_CONFIG['output_dir']) / 'task_distribution.png')
    plot_task_distribution(train_raw, save_path=auto_save)
else:
    print("train_raw chưa sẵn sàng. Hãy chạy cell Prepare Training Data trước để vẽ task distribution.")

In [ ]:
# Visualize 3: Model Architecture & Parameters
def plot_model_architecture(save_path=None):
    """Visualize LoRA configuration and parameter distribution"""
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. LoRA Configuration
    config_data = {
        'Rank (r)': UNIFIED_LORA_CONFIG['r'],
        'Alpha (α)': UNIFIED_LORA_CONFIG['lora_alpha'],
        'Dropout': UNIFIED_LORA_CONFIG['lora_dropout'] * 100,
        'Target Modules': len(UNIFIED_LORA_CONFIG['target_modules'])
    }
    
    ax1.barh(list(config_data.keys()), list(config_data.values()), 
             color=['#4285F4', '#34A853', '#FBBC04', '#EA4335'], alpha=0.8, edgecolor='black')
    ax1.set_xlabel('Value', fontsize=11, fontweight='bold')
    ax1.set_title('LoRA Configuration', fontsize=13, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, (k, v) in enumerate(config_data.items()):
        ax1.text(v, i, f' {v:.1f}' if 'Dropout' in k else f' {int(v)}', 
                va='center', fontweight='bold', fontsize=10)
    
    # 2. Target Modules
    target_modules = UNIFIED_LORA_CONFIG['target_modules']
    module_colors = plt.cm.Set3(np.linspace(0, 1, len(target_modules)))
    
    ax2.barh(range(len(target_modules)), [1]*len(target_modules), 
             color=module_colors, edgecolor='black', alpha=0.8)
    ax2.set_yticks(range(len(target_modules)))
    ax2.set_yticklabels(target_modules, fontsize=10)
    ax2.set_xlabel('Module Enabled', fontsize=11, fontweight='bold')
    ax2.set_title('Target Modules (LoRA Applied)', fontsize=13, fontweight='bold')
    ax2.set_xlim([0, 1.2])
    ax2.grid(False)
    
    # 3. Parameter Comparison
    base_params = 1500  # Million parameters
    lora_params = 45    # Million trainable params
    frozen_params = base_params - lora_params
    
    params_data = {
        'Frozen\nParameters': frozen_params,
        'Trainable\nLoRA Parameters': lora_params
    }
    
    bars = ax3.bar(params_data.keys(), params_data.values(), 
                   color=['#E8EAED', '#4285F4'], alpha=0.8, edgecolor='black', linewidth=2)
    ax3.set_ylabel('Parameters (Million)', fontsize=11, fontweight='bold')
    ax3.set_title('Parameter Distribution', fontsize=13, fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # Add percentage labels
    total = sum(params_data.values())
    for bar, value in zip(bars, params_data.values()):
        height = bar.get_height()
        percentage = (value / total) * 100
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(value)}M\n({percentage:.1f}%)',
                ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    # 4. Unified vs Separate Adapters Comparison
    metrics = ['Storage\n(MB)', 'Latency\n(ms)', 'Load Time\n(s)', 'Memory\n(GB)']
    unified_values = [80, 125, 0.8, 2.5]
    separate_values = [320, 500, 4.0, 3.5]
    
    x = np.arange(len(metrics))
    width = 0.35
    
    bars1 = ax4.bar(x - width/2, unified_values, width, label='Unified Adapter',
                    color='#34A853', alpha=0.8, edgecolor='black')
    bars2 = ax4.bar(x + width/2, separate_values, width, label='4 Separate Adapters',
                    color='#EA4335', alpha=0.8, edgecolor='black')
    
    ax4.set_ylabel('Value', fontsize=11, fontweight='bold')
    ax4.set_title('Unified vs Separate Adapters', fontsize=13, fontweight='bold')
    ax4.set_xticks(x)
    ax4.set_xticklabels(metrics, fontsize=10)
    ax4.legend(fontsize=10, loc='upper left')
    ax4.grid(True, alpha=0.3, axis='y')
    
    # Add improvement percentages
    for i, (u, s) in enumerate(zip(unified_values, separate_values)):
        improvement = ((s - u) / s) * 100
        ax4.text(i, max(u, s) + 20, f'↓{improvement:.0f}%',
                ha='center', fontweight='bold', fontsize=9, color='green')
    
    plt.tight_layout()
    if save_path:
        import os
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f' Saved plot to: {save_path}')
    plt.show()
    
    # Print summary
    print("\n Model Architecture Summary:")
    print(f"  Base Model: {MODEL_NAME}")
    print(f"  Total Parameters: {base_params}M")
    print(f"  Trainable Parameters: {lora_params}M ({(lora_params/base_params)*100:.2f}%)")
    print(f"  LoRA Rank: {UNIFIED_LORA_CONFIG['r']}")
    print(f"  LoRA Alpha: {UNIFIED_LORA_CONFIG['lora_alpha']}")
    print(f"  Target Modules: {len(UNIFIED_LORA_CONFIG['target_modules'])}")
    print(f"\n Unified Adapter Advantages:")
    print("   75% smaller storage (80MB vs 320MB)")
    print("   75% faster inference (125ms vs 500ms)")
    print("   80% faster loading (0.8s vs 4s)")
    print("   29% less memory (2.5GB vs 3.5GB)")

# Auto-render when config is ready
if 'UNIFIED_LORA_CONFIG' in globals() and 'MODEL_NAME' in globals():
    save_file = None
    if 'TRAINING_CONFIG' in globals() and isinstance(TRAINING_CONFIG, dict):
        save_file = str(Path(TRAINING_CONFIG['output_dir']) / 'model_architecture_summary.png')
    plot_model_architecture(save_path=save_file)
else:
    print("Thiếu MODEL_NAME hoặc UNIFIED_LORA_CONFIG. Hãy chạy các cell config/model trước.")

In [ ]:
# Merge LoRA weights into base model (optional for deployment)
def merge_unified_adapter(adapter_dir=None, merged_output_dir=None):
    """Merge unified LoRA adapter into Qwen3-1.7B base model."""
    assert 'qwen3' in MODEL_NAME.lower(), f'Merge target must be Qwen3, got: {MODEL_NAME}'
    from peft import PeftModel

    adapter_dir = adapter_dir or str(Path(TRAINING_CONFIG['output_dir']) / 'unified_lora_adapter')
    merged_output_dir = merged_output_dir or str(Path(TRAINING_CONFIG['output_dir']) / 'merged_qwen3_1_7b')

    print('Merging unified adapter into base model...')
    print(f'Base model : {MODEL_NAME}')
    print(f'Adapter dir: {adapter_dir}')
    print(f'Output dir : {merged_output_dir}')

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
        device_map='auto',
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    model = PeftModel.from_pretrained(model, adapter_dir)
    merged_model = model.merge_and_unload()

    out_path = Path(merged_output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    merged_model.save_pretrained(str(out_path))
    tokenizer.save_pretrained(str(out_path))

    print(f'Merged model saved to: {out_path}')
    print('Format: HuggingFace Transformers')
    return merged_model

print('='*60)
print('EXPORT OPTIONS (QWEN3-1.7B)')
print('='*60)
print('\n1. Keep LoRA Adapter (recommended for iteration):')
print('   - Small adapter package, fast retrain cycle')
print('   - Requires base Qwen3-1.7B + PEFT at runtime')
print('\n2. Merged Model (recommended for single artifact deploy):')
print('   - One standalone model folder for inference/deployment')
print('   - Run: merged_model = merge_unified_adapter()')

In [ ]:
# ============================================================================
# Final test evaluation: quality-gated metrics + useful charts + README report
# ============================================================================
import os
import re
import json
import warnings
import zipfile
import numpy as np
import pandas as pd
import jiwer
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from datetime import datetime
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, balanced_accuracy_score
from transformers.utils import logging as hf_logging

# Reduce repetitive generation warnings in long evaluation loops
hf_logging.set_verbosity_error()
warnings.filterwarnings('ignore', message='.*pad_token_id.*')

try:
    from bert_score import score as bertscore_score
    HAS_BERTSCORE = True
except Exception:
    HAS_BERTSCORE = False


def _safe_json_loads(text):
    if isinstance(text, dict):
        return text
    if not isinstance(text, str):
        return {}
    text = text.strip()
    if not text:
        return {}
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                return {}
    return {}


def _extract_gt_output(sample):
    out = sample.get('output', {})
    if isinstance(out, dict):
        return out
    if isinstance(out, str):
        d = _safe_json_loads(out)
        if d:
            return d
    messages = sample.get('messages', [])
    if isinstance(messages, list):
        for msg in reversed(messages):
            if isinstance(msg, dict) and msg.get('role') == 'assistant':
                d = _safe_json_loads(msg.get('content', ''))
                if d:
                    return d
    return {}


def _has_semantic_gt(task, gt_obj):
    if not isinstance(gt_obj, dict) or not gt_obj:
        return False
    if task == 'fluency':
        return 'fluency_score' in gt_obj
    if task == 'vocabulary':
        return 'level' in gt_obj
    if task == 'grammar':
        return 'corrected' in gt_obj
    if task == 'dialogue':
        return 'response' in gt_obj
    return False


TASK_PROMPT_NAME_MAP = {
    'fluency': 'fluency_scoring',
    'vocabulary': 'vocabulary_classification',
    'grammar': 'grammar_correction',
    'dialogue': 'dialogue_response',
}

def _build_user_prompt_from_sample(sample):
    task_raw = str(sample.get('task', 'unknown'))
    task_name = TASK_PROMPT_NAME_MAP.get(task_raw, task_raw)
    metadata = sample.get('metadata', {}) if isinstance(sample.get('metadata'), dict) else {}
    inp = sample.get('input')

    if not inp:
        msgs = sample.get('messages', [])
        if isinstance(msgs, list):
            for msg in msgs:
                if isinstance(msg, dict) and msg.get('role') == 'user':
                    inp = msg.get('content', '')
                    break
    inp = inp or ''

    if task_raw in ('fluency', 'vocabulary', 'grammar'):
        return f"Task: {task_name}\nText: {inp}"

    history = metadata.get('history') or metadata.get('conversation_history') or ''
    strategy = metadata.get('strategy') or 'socratic_questioning'
    return f"Task: {task_name}\nText: {inp}\nContext: {history}\nStrategy: {strategy}"


def _generate_prediction_json(model, tok, sample, max_new_tokens=128):
    user_prompt = _build_user_prompt_from_sample(sample)
    messages = [
        {'role': 'system', 'content': "You are LexiLingo's unified English tutor model. Respond ONLY valid JSON."},
        {'role': 'user', 'content': user_prompt},
    ]

    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors='pt')

    target_device = getattr(model, 'device', None)
    if target_device is not None:
        inputs = {k: v.to(target_device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )

    gen = tok.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return _safe_json_loads(gen), gen


def evaluate_on_testset(model, tok, test_raw_dataset, max_samples=None):
    if test_raw_dataset is None:
        raise ValueError('test_raw dataset is missing. Please provide test.jsonl.')

    n = len(test_raw_dataset)
    if max_samples is not None:
        n = min(n, int(max_samples))

    records = []
    for i in range(n):
        sample = test_raw_dataset[i]
        task = str(sample.get('task', 'unknown'))
        gt = _extract_gt_output(sample)
        pred_json, pred_text = _generate_prediction_json(model, tok, sample)

        records.append({
            'idx': i,
            'task': task,
            'gt': gt,
            'pred': pred_json,
            'pred_raw_text': pred_text,
            'parse_ok': bool(pred_json),
            'has_semantic_gt': _has_semantic_gt(task, gt),
        })

        if (i + 1) % 100 == 0:
            print(f'  evaluated {i+1}/{n} samples...')

    return records


def summarize_ground_truth_quality(test_raw_dataset, max_samples=None):
    n = len(test_raw_dataset)
    if max_samples is not None:
        n = min(n, int(max_samples))

    per_task_total = {}
    per_task_semantic = {}

    for i in range(n):
        s = test_raw_dataset[i]
        task = str(s.get('task', 'unknown'))
        gt = _extract_gt_output(s)

        per_task_total[task] = per_task_total.get(task, 0) + 1
        if _has_semantic_gt(task, gt):
            per_task_semantic[task] = per_task_semantic.get(task, 0) + 1

    summary = {}
    semantic_total = 0
    for task, total in per_task_total.items():
        semantic = per_task_semantic.get(task, 0)
        semantic_total += semantic
        summary[task] = {
            'samples': total,
            'semantic_gt_samples': semantic,
            'semantic_gt_coverage': float(semantic / total) if total else 0.0,
        }

    overall_cov = float(semantic_total / n) if n else 0.0
    return summary, overall_cov, n


def compute_task_metrics(records):
    task_groups = {}
    for r in records:
        task_groups.setdefault(r['task'], []).append(r)

    out = {}
    coverage = {}

    # Fluency: MAE
    if 'fluency' in task_groups:
        gts, preds = [], []
        for r in task_groups['fluency']:
            if 'fluency_score' in r['gt'] and 'fluency_score' in r['pred']:
                gts.append(float(r['gt']['fluency_score']))
                preds.append(float(r['pred']['fluency_score']))

        fluency_total = len(task_groups['fluency'])
        fluency_cov = float(len(gts) / fluency_total) if fluency_total else 0.0
        coverage['fluency'] = fluency_cov

        out['fluency'] = {
            'samples': fluency_total,
            'paired_samples': len(gts),
            'coverage': fluency_cov,
        }
        if gts:
            out['fluency']['MAE'] = float(np.mean(np.abs(np.array(gts) - np.array(preds))))

    # Vocabulary: Acc/F1/Precision/Recall
    if 'vocabulary' in task_groups:
        y_true, y_pred = [], []
        for r in task_groups['vocabulary']:
            gt_level = r['gt'].get('level')
            pd_level = r['pred'].get('level')
            if gt_level is not None and pd_level is not None:
                y_true.append(str(gt_level))
                y_pred.append(str(pd_level))

        vocab_total = len(task_groups['vocabulary'])
        vocab_cov = float(len(y_true) / vocab_total) if vocab_total else 0.0
        coverage['vocabulary'] = vocab_cov

        out['vocabulary'] = {
            'samples': vocab_total,
            'paired_samples': len(y_true),
            'coverage': vocab_cov,
        }
        if y_true:
            f1_macro = float(f1_score(y_true, y_pred, average='macro', zero_division=0))
            bal_acc = float(balanced_accuracy_score(y_true, y_pred))
            out['vocabulary'].update({
                'Accuracy': float(accuracy_score(y_true, y_pred)),
                'Balanced_Accuracy': bal_acc,
                'F1_macro': f1_macro,
                'F1_weighted': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
                'Precision_macro': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
                'Precision_weighted': float(precision_score(y_true, y_pred, average='weighted', zero_division=0)),
                'Recall_macro': float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
                'Recall_weighted': float(recall_score(y_true, y_pred, average='weighted', zero_division=0)),
                'F1_macro_coverage_adjusted': float(f1_macro * vocab_cov),
                'Balanced_Accuracy_coverage_adjusted': float(bal_acc * vocab_cov),
                'labels': sorted(set(y_true) | set(y_pred)),
                'y_true': y_true,
                'y_pred': y_pred,
            })

    # Grammar: WER + exact match
    if 'grammar' in task_groups:
        gt_texts, pred_texts = [], []
        for r in task_groups['grammar']:
            gt = r['gt'].get('corrected')
            pd = r['pred'].get('corrected')
            if gt is not None and pd is not None:
                gt_texts.append(str(gt))
                pred_texts.append(str(pd))

        grammar_total = len(task_groups['grammar'])
        grammar_cov = float(len(gt_texts) / grammar_total) if grammar_total else 0.0
        coverage['grammar'] = grammar_cov

        out['grammar'] = {
            'samples': grammar_total,
            'paired_samples': len(gt_texts),
            'coverage': grammar_cov,
        }
        if gt_texts:
            out['grammar'].update({
                'WER': float(jiwer.wer(gt_texts, pred_texts)),
                'CER': float(jiwer.cer(gt_texts, pred_texts)),
                'ExactMatch': float(np.mean([1.0 if g.strip() == p.strip() else 0.0 for g, p in zip(gt_texts, pred_texts)])),
            })

    # Dialogue: BERTScore (F1)
    if 'dialogue' in task_groups:
        gt_resp, pred_resp = [], []
        for r in task_groups['dialogue']:
            gt = r['gt'].get('response')
            pd = r['pred'].get('response')
            if gt is not None and pd is not None:
                gt_resp.append(str(gt))
                pred_resp.append(str(pd))

        dialogue_total = len(task_groups['dialogue'])
        dialogue_cov = float(len(gt_resp) / dialogue_total) if dialogue_total else 0.0
        coverage['dialogue'] = dialogue_cov

        out['dialogue'] = {
            'samples': dialogue_total,
            'paired_samples': len(gt_resp),
            'coverage': dialogue_cov,
        }
        if gt_resp:
            if HAS_BERTSCORE:
                P, R, F1 = bertscore_score(pred_resp, gt_resp, lang='en', verbose=False)
                out['dialogue']['BERTScore_P'] = float(P.mean().item())
                out['dialogue']['BERTScore_R'] = float(R.mean().item())
                out['dialogue']['BERTScore_F1'] = float(F1.mean().item())
            else:
                out['dialogue']['BERTScore'] = 'not_available_install_bert_score'

            out['dialogue']['ExactMatch'] = float(np.mean([1.0 if g.strip() == p.strip() else 0.0 for g, p in zip(gt_resp, pred_resp)]))

    # Parse success by task
    parse_by_task = {}
    for task, rows in task_groups.items():
        parse_by_task[task] = float(np.mean([1.0 if r['parse_ok'] else 0.0 for r in rows])) if rows else 0.0

    out['parse_success_rate_by_task'] = parse_by_task
    out['parse_success_rate_overall'] = float(np.mean([1.0 if r['parse_ok'] else 0.0 for r in records])) if records else 0.0

    # Semantic GT coverage by task
    out['semantic_gt_coverage_by_task'] = coverage
    out['semantic_gt_coverage_overall'] = float(np.mean(list(coverage.values()))) if coverage else 0.0

    out['evaluation_mode'] = 'semantic+parse' if out['semantic_gt_coverage_overall'] > 0 else 'parse-only'

    return out


def save_useful_eval_plots(metrics, report_dir):
    report_dir = Path(report_dir)
    report_dir.mkdir(parents=True, exist_ok=True)
    paths = {}

    # Plot 1: key metric per task
    task_metric = {}
    if 'fluency' in metrics and 'MAE' in metrics['fluency']:
        task_metric['fluency_MAE(lower_better)'] = metrics['fluency']['MAE']
    if 'vocabulary' in metrics and 'F1_macro' in metrics['vocabulary']:
        task_metric['vocabulary_F1'] = metrics['vocabulary']['F1_macro']
    if 'grammar' in metrics and 'WER' in metrics['grammar']:
        task_metric['grammar_WER(lower_better)'] = metrics['grammar']['WER']
    if 'dialogue' in metrics and 'BERTScore_F1' in metrics['dialogue']:
        task_metric['dialogue_BERTScore_F1'] = metrics['dialogue']['BERTScore_F1']

    if task_metric:
        plt.figure(figsize=(11, 5))
        x = list(task_metric.keys())
        y = list(task_metric.values())
        plt.bar(x, y)
        plt.title('Key Test Metrics by Task')
        plt.ylabel('Metric Value')
        plt.xticks(rotation=20, ha='right')
        plt.grid(axis='y', alpha=0.2)
        p = report_dir / 'test_key_metrics.png'
        plt.tight_layout()
        plt.savefig(p, dpi=250)
        plt.show()
        plt.close()
        paths['test_key_metrics'] = str(p)

    # Plot 2: vocabulary confusion matrix
    if 'vocabulary' in metrics and 'y_true' in metrics['vocabulary'] and metrics['vocabulary']['paired_samples'] > 0:
        y_true = metrics['vocabulary']['y_true']
        y_pred = metrics['vocabulary']['y_pred']
        labels = metrics['vocabulary']['labels']
        cm = confusion_matrix(y_true, y_pred, labels=labels)
        plt.figure(figsize=(7, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
        plt.title('Vocabulary Confusion Matrix (Test)')
        plt.xlabel('Predicted')
        plt.ylabel('Ground Truth')
        p = report_dir / 'vocabulary_confusion_matrix.png'
        plt.tight_layout()
        plt.savefig(p, dpi=250)
        plt.show()
        plt.close()
        paths['vocabulary_confusion_matrix'] = str(p)

    return paths


def export_readme_report(metrics, plot_paths, records, report_dir, gt_quality_summary, gt_overall_cov):
    report_dir = Path(report_dir)
    report_dir.mkdir(parents=True, exist_ok=True)

    metrics_json = report_dir / 'test_metrics.json'
    with open(metrics_json, 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    pred_json = report_dir / 'test_predictions.json'
    with open(pred_json, 'w', encoding='utf-8') as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

    gt_json = report_dir / 'ground_truth_quality.json'
    with open(gt_json, 'w', encoding='utf-8') as f:
        json.dump({
            'overall_semantic_gt_coverage': gt_overall_cov,
            'by_task': gt_quality_summary,
        }, f, indent=2, ensure_ascii=False)

    readme = report_dir / 'README_TEST_EVALUATION.md'
    lines = []
    lines.append('# Test Evaluation Report')
    lines.append('')
    lines.append(f'- Generated at: {datetime.now().isoformat()}')
    lines.append(f'- Model: {MODEL_NAME}')
    lines.append(f"- Output dir: {TRAINING_CONFIG.get('output_dir')}")
    lines.append(f"- Evaluation mode: {metrics.get('evaluation_mode', 'unknown')}")
    lines.append(f"- Overall semantic GT coverage: {gt_overall_cov:.4f}")
    lines.append('')

    lines.append('## Ground Truth Quality Check')
    lines.append('')
    for task, stat in sorted(gt_quality_summary.items()):
        lines.append(
            f"- {task}: semantic_gt={stat['semantic_gt_samples']}/{stat['samples']} "
            f"({stat['semantic_gt_coverage']:.4f})"
        )
    lines.append('')

    if gt_overall_cov < 0.50:
        lines.append('## Warning')
        lines.append('')
        lines.append('- Semantic ground-truth coverage is low (<0.50).')
        lines.append('- Most semantic metrics may be noisy or not representative.')
        lines.append('- Parse success is still valid as output-format stability indicator.')
        lines.append('')

    lines.append('## Task Metrics')
    lines.append('')
    for task in ['fluency', 'vocabulary', 'grammar', 'dialogue']:
        if task in metrics:
            lines.append(f'### {task}')
            for k, v in metrics[task].items():
                if k in ('y_true', 'y_pred', 'labels'):
                    continue
                lines.append(f'- {k}: {v}')
            lines.append('')

    if 'parse_success_rate_by_task' in metrics:
        lines.append('## Parse Success Rate by Task')
        lines.append('')
        for k, v in metrics['parse_success_rate_by_task'].items():
            lines.append(f'- {k}: {v:.4f}')
        lines.append('')

    if 'semantic_gt_coverage_by_task' in metrics:
        lines.append('## Semantic GT Coverage by Task')
        lines.append('')
        for k, v in metrics['semantic_gt_coverage_by_task'].items():
            lines.append(f'- {k}: {v:.4f}')
        lines.append('')

    lines.append('## Plots')
    lines.append('')
    for name, p in plot_paths.items():
        rel = Path(p).name
        lines.append(f'### {name}')
        lines.append(f'![{name}]({rel})')
        lines.append('')

    lines.append('## Artifacts')
    lines.append('')
    lines.append('- test_metrics.json')
    lines.append('- test_predictions.json')
    lines.append('- ground_truth_quality.json')

    readme.write_text('\n'.join(lines), encoding='utf-8')

    # zip report for downloading
    zip_path = report_dir / 'test_evaluation_bundle.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(readme, arcname=readme.name)
        zf.write(metrics_json, arcname=metrics_json.name)
        zf.write(pred_json, arcname=pred_json.name)
        zf.write(gt_json, arcname=gt_json.name)
        for _, p in plot_paths.items():
            pth = Path(p)
            if pth.exists():
                zf.write(pth, arcname=pth.name)

    return readme, metrics_json, pred_json, gt_json, zip_path


# ----------------------------
# Run final test evaluation
# ----------------------------
if 'test_raw' not in globals() or test_raw is None:
    raise ValueError('test_raw is missing. Please ensure test.jsonl is loaded in data cell.')

if 'unified_model' in globals() and unified_model is not None:
    eval_model = unified_model
elif 'trainer' in globals() and trainer is not None:
    eval_model = trainer.model
else:
    raise ValueError('No trained model found. Run the training cell first.')

MAX_TEST_SAMPLES = 300  # default quick evaluation; set None for full test set
MIN_SEMANTIC_GT_COVERAGE = 0.50
ALLOW_PARSE_ONLY_EVAL = True

gt_quality_summary, gt_overall_cov, checked_n = summarize_ground_truth_quality(
    test_raw, max_samples=MAX_TEST_SAMPLES
 )

print('\n' + '='*80)
print('GROUND TRUTH QUALITY PRECHECK')
print('='*80)
print(f'Samples to evaluate: {checked_n}')
print(f'Overall semantic GT coverage: {gt_overall_cov:.4f}')
for task, stat in sorted(gt_quality_summary.items()):
    print(
        f"  - {task}: {stat['semantic_gt_samples']}/{stat['samples']} "
        f"({stat['semantic_gt_coverage']:.4f})"
    )
print('='*80)

if gt_overall_cov < MIN_SEMANTIC_GT_COVERAGE and not ALLOW_PARSE_ONLY_EVAL:
    raise ValueError(
        f'Semantic GT coverage too low ({gt_overall_cov:.4f} < {MIN_SEMANTIC_GT_COVERAGE}). '
        'Set ALLOW_PARSE_ONLY_EVAL=True to continue parse-focused evaluation.'
    )

records = evaluate_on_testset(eval_model, tokenizer, test_raw, max_samples=MAX_TEST_SAMPLES)
metrics = compute_task_metrics(records)

report_dir = Path(TRAINING_CONFIG['output_dir']) / 'test_report'
plot_paths = save_useful_eval_plots(metrics, report_dir)
readme_path, metrics_path, preds_path, gt_quality_path, zip_path = export_readme_report(
    metrics,
    plot_paths,
    records,
    report_dir,
    gt_quality_summary,
    gt_overall_cov,
 )

print('\n' + '='*80)
print(' TEST EVALUATION COMPLETED')
print('='*80)
print(f'README: {readme_path}')
print(f'METRICS: {metrics_path}')
print(f'PREDICTIONS: {preds_path}')
print(f'GT QUALITY: {gt_quality_path}')
print(f'ZIP BUNDLE: {zip_path}')
print('='*80)

# Auto-download on Colab
try:
    from google.colab import files
    files.download(str(zip_path))
    for _, p in plot_paths.items():
        files.download(str(p))
except Exception:
    print(' Not running in Colab download context. Files are saved on disk.')